# OpenCDR — Complete Manual Test Notebook

**Purpose.** A single, runnable, ordered checklist for verifying a live OpenCDR deployment end to
end — every subsystem, not just the core loop — against one AWS account/stage. Written for the
same two audiences as [`docs/manual-testing.md`](../docs/manual-testing.md) (the runbook this
notebook expands into an exhaustive, executable form): the person who just deployed it, verifying
their own work, and a client verifying what they were handed.

**What this is not.** A substitute for `pytest tests/` (unit correctness) or CI's own
`post-deploy-*` jobs (gated on every push to `main`). Those prove the code is correct in the
abstract. This notebook proves *this specific deployment, in this specific AWS account,* actually
detects, correlates, notifies, responds, and rolls back — the class of bug that ships past 100%
unit test coverage and only surfaces against a real account (see the project's own roadmap notes
referenced in `docs/manual-testing.md`).

**How to use it.**
1. Run the *Setup* cells once, filling in your stage's API URL/key.
2. Work through the phases **in order** — they're ordered by blast radius if skipped, not
   alphabetically. A failure in an early phase invalidates a "pass" in a later one (e.g. there's no
   point checking notification delivery if detection itself is broken).
3. Fully automated checks call `record(...)` for you and print ✅/❌. Checks that need a human eye
   (a Slack message arriving, a UI page rendering) call `manual_check(...)` — read the criteria,
   verify it yourself, then edit the following line to `record(phase, check, "PASS")` (or `"FAIL"`)
   and re-run the cell.
4. The final cell prints a summary table across every recorded check and an overall verdict.

**Architecture reference**, if anything below is unfamiliar: [`docs/architecture.md`](../docs/architecture.md).


## Priority legend

Phases are ordered by **what breaks downstream if this phase is broken**, not by how interesting
the subsystem is. Treat P0/P1 failures as blocking; P2 as high-priority but not launch-blocking on
its own; P3/P4 as expected to be skipped entirely for deployments that don't use that feature
(single-account, single-region, no automated response armed, etc.).

| Priority | Meaning | What a failure here means |
|---|---|---|
| **P0 — Blocker** | The core event→signal pipeline | Nothing else in this document can be trusted until this passes. Stop and fix before continuing. |
| **P1 — Critical** | What makes a detection *useful* to a human | Detections happen but nobody finds out, or an armed automated response can't be trusted / undone. |
| **P2 — High** | The management/API surface and detection breadth | The product works for the golden path but a client managing it directly, or an edge-case rule, may be broken. |
| **P3 — Medium** | Multi-region / multi-account / operational visibility | Real gaps, but scoped to deployments that use these features, or coverage of failures rather than the failures themselves. |
| **P4 — Low / Optional** | Retention, deep security posture, SIEM fan-out | Worth doing once per deployment; not worth re-running on every change. |

**Result icons:** ✅ PASS · ❌ FAIL · ⏭️ SKIP (not applicable to this deployment) · ⚠️ WARN (passed but worth a second look) · 👤 MANUAL (needs a human to confirm)


In [ ]:
import json, os, re, subprocess, time, urllib.request, urllib.error
from datetime import datetime, timezone

RESULTS = []  # {"phase", "check", "status", "detail"}

_ICONS = {"PASS": "✅", "FAIL": "❌", "SKIP": "⏭️", "WARN": "⚠️",
          "MANUAL-PENDING": "\U0001f464"}

def record(phase, check, status, detail=""):
    RESULTS.append({"phase": phase, "check": check, "status": status, "detail": detail})
    icon = _ICONS.get(status, "•")
    print(f"{icon} [{phase}] {check} -> {status}" + (f" — {detail}" if detail else ""))
    return status

def manual_check(phase, check, criteria):
    print(f"\U0001f464 MANUAL CHECK [{phase}] {check}")
    print(f"   Expected: {criteria}")
    print(f"   -> After verifying by hand, change the record(...) call below this print "
          f"block to 'PASS' or 'FAIL' and re-run this cell.")
    record(phase, check, "MANUAL-PENDING", criteria)

def api(method, path, body=None, key=None, expect=None, quiet=False):
    '''Call the OpenCDR REST API directly (no CLI dependency) and return (status, json_body).'''
    url = os.environ["OPENCDR_API_URL"].rstrip("/") + path
    headers = {"x-api-key": key or os.environ["OPENCDR_API_KEY"]}
    data = None
    if body is not None:
        data = json.dumps(body).encode()
        headers["Content-Type"] = "application/json"
    req = urllib.request.Request(url, data=data, headers=headers, method=method)
    try:
        with urllib.request.urlopen(req, timeout=20) as resp:
            status = resp.status
            raw = resp.read()
            payload = json.loads(raw) if raw else {}
    except urllib.error.HTTPError as e:
        status = e.code
        try:
            payload = json.loads(e.read())
        except Exception:
            payload = {}
    if not quiet:
        print(f"{method} {path} -> {status}")
    if expect is not None and status != expect:
        print(f"   !! unexpected status {status} (wanted {expect}): {json.dumps(payload)[:500]}")
    return status, payload

def sh(cmd, check_rc=True, quiet=False):
    '''Run a shell command, streaming stdout/stderr, returning the CompletedProcess.'''
    if not quiet:
        print(f"$ {cmd}")
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if p.stdout:
        print(p.stdout)
    if p.stderr:
        print(p.stderr)
    if check_rc and p.returncode != 0:
        raise RuntimeError(f"command failed (rc={p.returncode}): {cmd}")
    return p

def count_markers(text):
    '''Best-effort PASS/FAIL/MISS counter for scripts/opencdr.py test output.'''
    return {
        "PASS": len(re.findall(r"\bPASS\b", text)),
        "FAIL": len(re.findall(r"\bFAIL\b", text)),
        "MISS": len(re.findall(r"\bMISS\b", text)),
    }

print("Helpers loaded.")


---
## Phase 0 — Prerequisites & Environment (do this once per stage)

Not a "phase" in the priority sense above — nothing below runs without this. Covers the
prerequisites from [`docs/setup.md`](../docs/setup.md) and
[`docs/manual-testing.md`](../docs/manual-testing.md#prerequisites).

- AWS CLI v2, credentials for the **target account** (`aws sts get-caller-identity` succeeds)
- `jq`, Python 3.12 with `requirements-dev.txt` installed
- CloudTrail **enabled** in the target account/region (management events, read+write) — without
  this, `processor` never receives anything, and every phase below will fail for a reason that has
  nothing to do with OpenCDR itself
- The deployment's API URL and an all-scopes API key (deploy output, or SSM: `/opencdr-<stage>/api-url`,
  `/opencdr-<stage>/api-key`)
- Run this notebook's kernel from the repo root (`opencdr-internal/`) so relative script paths resolve


In [ ]:
# --- Fill these in for the stage you're testing, then run this cell ---
os.environ["OPENCDR_STAGE"] = "dev"                 # --stage passed to serverless/scripts
os.environ["AWS_REGION"] = "us-east-1"              # region the stack is deployed in
os.environ["OPENCDR_API_URL"] = "<PASTE the https://xxxx.execute-api.<region>.amazonaws.com/<stage> URL>"
os.environ["OPENCDR_API_KEY"] = "<PASTE the all-scopes API key>"

# Optional, only needed for the Phase 4 scope-enforcement check: a key minted with ONLY the
# 'read' scope (see docs/api-reference.md#api-key-scopes). Leave blank to skip that one check.
os.environ.setdefault("OPENCDR_READONLY_API_KEY", "")

print("stage      :", os.environ["OPENCDR_STAGE"])
print("region     :", os.environ["AWS_REGION"])
print("api url set:", not os.environ["OPENCDR_API_URL"].startswith("<"))
print("api key set:", not os.environ["OPENCDR_API_KEY"].startswith("<"))


In [ ]:
# --- Tooling & AWS identity ---
sh("aws sts get-caller-identity")
sh("jq --version")
sh("python3 --version")
identity = json.loads(sh('aws sts get-caller-identity --output json', quiet=True).stdout)
record("P0-Setup", "AWS credentials resolve", "PASS", identity.get("Account"))


In [ ]:
# --- Detection rules submodule actually populated (a common silent-zero-rules cause) ---
p = sh("git submodule status support_files/detection_rules", check_rc=False)
populated = len(list(__import__('pathlib').Path('support_files/detection_rules').rglob('*.json'))) > 0
record("P0-Setup", "detection_rules submodule populated", "PASS" if populated else "FAIL",
       "run: git submodule update --init" if not populated else "")


In [ ]:
# --- CloudTrail enabled in the target account/region ---
# Trail name varies per deployment, so this is listed rather than auto-asserted.
sh(f'aws cloudtrail describe-trails --region "$AWS_REGION" --query "trailList[].{{Name:Name,Multi:IsMultiRegionTrail,Home:HomeRegion}}" --output table', check_rc=False)
manual_check("P0-Setup", "CloudTrail is logging",
             "At least one trail above has management-event logging ON in this region "
             "(aws cloudtrail get-trail-status --name <trail> --query IsLogging).")
# record("P0-Setup", "CloudTrail is logging", "PASS")


In [ ]:
# --- Point the CLI at this stage (same logic CI/MCP server use) ---
sh('python3 scripts/opencdr.py config set --url "$OPENCDR_API_URL" --key "$OPENCDR_API_KEY"')


---
## Phase 1 — P0 (Blocker): The Core Detection Pipeline

*"a real event, in this account, produces a real signal."* Everything else in this notebook
assumes this phase passes. If anything here fails, stop and fix it before proceeding —
[`docs/architecture.md`](../docs/architecture.md) for how `processor`/`signalWriter` fit together.

| # | Check |
|---|---|
| 1.1 | API reachable, key valid |
| 1.2 | `/help` route list matches what this deployment actually serves |
| 1.3 | Detection rules are loaded |
| 1.4 | A raw event, sent to `processor`, produces a signal (the core loop) |
| 1.5 | Signals and logs are independently queryable (read path) |


In [ ]:
# 1.1 — API reachable, key valid
status, body = api("GET", "/status", expect=200)
ok = status == 200 and "service" in body and "time" in body
record("P1-Core", "API /status healthy", "PASS" if ok else "FAIL", json.dumps(body)[:300])


In [ ]:
# 1.2 — /help lists the routes this deployment actually serves (no auth required)
status, body = api("GET", "/help", expect=200)
help_text = json.dumps(body)
expected_substrings = ["/signals", "/logs", "/rules", "/settings", "/ir-roles", "/ir-actions"]
missing = [s for s in expected_substrings if s not in help_text]
record("P1-Core", "/help lists expected routes", "PASS" if not missing else "WARN",
       f"missing: {missing}" if missing else "all core routes present")
print(help_text[:1000])


In [ ]:
# 1.3 — Detection rules are loaded
status, body = api("GET", "/rules?rule_kind=signal", expect=200)
signal_rules = body.get("items", body if isinstance(body, list) else [])
n_signal = len(signal_rules) if isinstance(signal_rules, list) else 0
record("P1-Core", "signal rules loaded", "PASS" if n_signal > 0 else "FAIL",
       f"{n_signal} signal rules" if n_signal else "run: ./scripts/load_rules.sh --stage $OPENCDR_STAGE (see docs/detection-rules.md)")


In [ ]:
# 1.4 — THE CORE LOOP: send every fixture in support_files/test_events/ to the deployed
# processor Lambda and confirm a matching signal lands in signals-table-v2.
# Expect PASS for every fixture except 028_guardduty_catchall (deliberately has no dedicated
# fixture -- see docs/detection-rules.md). A MISS almost always means step 1.3 wasn't done
# against THIS stage/region.
result = sh('python3 scripts/opencdr.py test deployed --stage "$OPENCDR_STAGE" --region "$AWS_REGION"',
            check_rc=False)
counts = count_markers(result.stdout)
print(counts)
# Allow exactly one MISS (028_guardduty_catchall) and zero FAILs.
ok = counts["FAIL"] == 0 and counts["MISS"] <= 1
record("P1-Core", "detection fires end-to-end (test deployed)", "PASS" if ok else "FAIL",
       f"{counts} (expected FAIL=0, MISS<=1)")


In [ ]:
# Narrow to a single fixture while debugging a MISS/FAIL from above, e.g.:
# sh('python3 scripts/opencdr.py test deployed --stage "$OPENCDR_STAGE" --event 011')


In [ ]:
# 1.5 — Signals and logs are queryable independently of the write path above
status, sig_body = api("GET", "/signals?severity=HIGH&page_size=5", expect=200)
sig_items = sig_body.get("items", [])
record("P1-Core", "signals readable via /signals", "PASS" if isinstance(sig_items, list) else "FAIL",
       f"{len(sig_items)} HIGH signals in default 7-day window")

status, log_body = api("GET", "/logs?service=OPENCDR-PROCESSOR&page_size=5", expect=200)
log_items = log_body.get("items", [])
record("P1-Core", "logs readable via /logs", "PASS" if isinstance(log_items, list) else "FAIL",
       f"{len(log_items)} OPENCDR-PROCESSOR log lines")

# Stash one signal's detection_id as a "canary" -- reused in Phase 8 to confirm archival.
status, any_sig = api("GET", "/signals?severity=HIGH&page_size=1&order=desc", expect=200, quiet=True)
_items = any_sig.get("items", [])
CANARY_DETECTION_ID = _items[0].get("detection_id") if _items else None
print("Canary detection_id for later phases:", CANARY_DETECTION_ID)


---
## Phase 2 — P1 (Critical): Alert Delivery

A detection nobody hears about might as well not have happened. There's no synthetic
"send me a test notification" endpoint — the only real proof is a live detection reaching a real
channel. See [`docs/notifications.md`](../docs/notifications.md).


In [ ]:
# 2.1 — Current channel configuration (secrets come back masked -- see docs/security.md)
status, settings = api("GET", "/settings", expect=200)
print(json.dumps(settings, indent=2))
configured_channels = [c for c in ("slack", "discord", "email", "securityhub", "jira", "webhook")
                       if settings.get(c)]
record("P2-Notify", "at least one channel configured", "PASS" if configured_channels else "FAIL",
       f"configured: {configured_channels}" if configured_channels
       else "run: python3 scripts/opencdr.py setup")


In [ ]:
# 2.2 — Configure a channel if none is set (uncomment + fill in ONE of these), then re-run 2.1:
# sh('python3 scripts/opencdr.py settings set --slack-webhook "<SLACK_WEBHOOK_URL>"')
# sh('python3 scripts/opencdr.py settings set --discord-webhook "<DISCORD_WEBHOOK_URL>"')

# Fire a fixture whose rule has notify:true (001 = console login without MFA) at the configured channel(s).
sh('python3 scripts/opencdr.py test deployed --stage "$OPENCDR_STAGE" --event 001', check_rc=False)
manual_check("P2-Notify", "notification lands in configured channel",
             "The 001 (console login, no MFA) alert appears in the channel(s) configured above "
             "within a few seconds. If not, check: python3 scripts/opencdr.py logs list "
             "--service OPENCDR-NOTIFIER --page-size 10")
# record("P2-Notify", "notification lands in configured channel", "PASS")


In [ ]:
# 2.3 -- Per-severity routing (docs/notifications.md#per-severity-routing) -- config sanity only;
# actually exercising two different severities into two different channels is a manual follow-up
# if you rely on this feature.
routing = settings.get("routing", {})
print("routing:", json.dumps(routing, indent=2) or "(none configured -- falls back to auto-fan-out to every enabled channel)")
record("P2-Notify", "routing config readable", "PASS")


In [ ]:
# 2.4 -- GuardDuty notifications default OFF unless explicitly opted in
# (docs/notifications.md#guardduty-notifications) -- absence of this key means EVERY GuardDuty
# item is silently skipped, including CRITICAL ones. Confirm it's a deliberate choice, not an
# oversight, for this deployment.
gd = settings.get("guardduty_notify")
if gd is None:
    record("P2-Notify", "guardduty_notify is a deliberate choice", "WARN",
           "key absent entirely -- ALL GuardDuty findings (including CRITICAL) are silently "
           "not notified. Fine if intentional, otherwise: "
           "opencdr.py settings set --guardduty-notify-default false --guardduty-notify-severity CRITICAL=true")
else:
    print(json.dumps(gd, indent=2))
    record("P2-Notify", "guardduty_notify is a deliberate choice", "PASS")


In [ ]:
# 2.5 -- The dashboard (UI) connects and shows real data. UI lives in a separate repo
# (opencdr-ui-internal) -- can't be automated from this notebook.
manual_check("P2-Notify", "UI Overview page",
             "Connection panel shows the same service/time as Phase 1.1. Signals-by-severity "
             "widget reflects Phase 1's signals; switching Today/7d/30d changes counts plausibly.")
manual_check("P2-Notify", "UI Signals/Logs pages",
             "Rows from Phase 1 appear; row click opens a detail modal; column-header sort works "
             "and defaults to time descending.")
manual_check("P2-Notify", "UI Rules page",
             f"Rule count matches Phase 1.3's {n_signal if 'n_signal' in dir() else '?'} signal rules "
             "(plus correlation/list rules).")


---
## Phase 3 — P1 (Critical): Automated Response & Rollback

> ⚠️ **This phase takes a real, if synthetic-triggered, action in the target AWS account.**
> Only proceed if at least one loaded rule has `response_module` set (rules load with it **stripped
> by default** -- see [`docs/detection-rules.md`](../docs/detection-rules.md) -- so this requires
> having run `load_rules.sh --with-response-modules` deliberately) and you've read
> [Response modules](../docs/incident-response.md#response-modules). Skip this entire phase in an
> account you're not prepared to see IAM/network changes in.


In [ ]:
# 3.1 -- What mode is this deployment actually armed in? CI defaults to LIVE (DREDGE_DRY_RUN=false)
# on every push to main -- serverless.yml's own dry-run default only applies to a bare local
# `serverless deploy` with nothing exported. Don't assume dry-run; check.
for fn in ("responder", "rollbackHandler"):
    p = sh(f'aws lambda get-function-configuration --function-name "opencdr-$OPENCDR_STAGE-{fn}" '
           f'--region "$AWS_REGION" --query "Environment.Variables.DREDGE_DRY_RUN" --output text',
           check_rc=False, quiet=True)
    mode = (p.stdout or "").strip()
    print(f"{fn}: DREDGE_DRY_RUN={mode!r}")
    record("P3-IR", f"{fn} DREDGE_DRY_RUN known before testing", "WARN" if mode in ("false", "") else "PASS",
           f"mode={mode!r} -- 'false'/empty means LIVE, real AWS calls will be made below")


**Gate.** Only continue past this point if:
1. You intentionally loaded rules with `--with-response-modules`, **and**
2. You're comfortable with the mode reported in 3.1 (dry-run for a rehearsal, or live if this
   account is meant to receive real containment actions), **and**
3. You've picked a fixture whose rule has a `response_module` you understand
   (e.g. `011_security_group_opened` -> `deauthorize_security_group_rules`).

If any of those isn't true, skip to [Phase 4](#Phase-4).


In [ ]:
# 3.2 -- Trigger a response_module rule and confirm the action lands as a rollback-eligible IR Action.
RESPONSE_FIXTURE = "011"  # change to match a response_module rule you've actually armed
sh(f'python3 scripts/opencdr.py test deployed --stage "$OPENCDR_STAGE" --event {RESPONSE_FIXTURE}', check_rc=False)
time.sleep(5)  # give responder a moment to drain the responses queue
status, actions = api("GET", "/ir-actions?page_size=5", expect=200)
items = actions.get("items", [])
print(json.dumps(items[:1], indent=2))
found = bool(items)
record("P3-IR", "response_module action recorded", "PASS" if found else "FAIL",
       f"{len(items)} IR action(s); check logs list --service OPENCDR-RESPONDER if empty")
IR_DETECTION_ID = items[0]["detection_id"] if items else None
print("IR_DETECTION_ID for rollback test:", IR_DETECTION_ID)


In [ ]:
# 3.3 -- Roll it back and poll until the async rollback settles.
if IR_DETECTION_ID:
    status, resp = api("POST", f"/ir-actions/{IR_DETECTION_ID}/rollback", expect=202)
    print(json.dumps(resp, indent=2))
    final = None
    for _ in range(12):  # up to ~60s
        time.sleep(5)
        _, action = api("GET", f"/ir-actions/{IR_DETECTION_ID}", expect=200, quiet=True)
        rb = action.get("rollback_status")
        print("rollback_status:", rb)
        if rb in ("succeeded", "failed"):
            final = action
            break
    if final is None:
        record("P3-IR", "rollback settles within ~60s", "FAIL", "still pending -- check rollbackHandler logs")
    elif final.get("rollback_status") == "succeeded":
        record("P3-IR", "rollback settles within ~60s", "PASS")
    else:
        record("P3-IR", "rollback settles within ~60s", "FAIL",
               final.get("rollback_error", "")[:300] +
               " -- see docs/ir-role.md if this is an AssumeRole AccessDenied")
else:
    record("P3-IR", "rollback settles within ~60s", "SKIP", "no IR action from 3.2")


In [ ]:
# 3.4 -- Remediation-success and rollback-success notifications (docs/incident-response.md#remediation-notifications)
manual_check("P3-IR", "remediation-success notification (blue-grey if dry-run, green if live)",
             "A second, distinct notification for the 3.2 action appears in the configured "
             "channel(s) -- green 'remediation succeeded', or blue-grey 'DRY RUN -- SIMULATED' "
             "if 3.1 reported dry-run mode.")
manual_check("P3-IR", "rollback-success notification (purple)",
             "A third, purple-styled notification for the 3.3 rollback appears -- distinct from "
             "both the original alert and the green remediation-success one.")


In [ ]:
# 3.5 -- Rate-limit / circuit breaker config sanity (NOT a live trip test -- tripping it on
# purpose in a real account is disruptive and not worth the risk for a routine test pass).
p = sh('aws lambda get-function-configuration --function-name "opencdr-$OPENCDR_STAGE-responder" '
       '--region "$AWS_REGION" --query "Environment.Variables.{MaxActions:RESPONDER_RATE_LIMIT_MAX_ACTIONS,WindowMin:RESPONDER_RATE_LIMIT_WINDOW_MINUTES}" '
       '--output json', check_rc=False, quiet=True)
print(p.stdout)
record("P3-IR", "rate-limit config present", "PASS" if p.stdout.strip() not in ("", "{}", "null") else "WARN",
       "defaults (20/5min) apply if unset -- see docs/incident-response.md#rate-limiting")


In [ ]:
# 3.6 -- Multi-account only: IR role trust policy must list BOTH responder's and
# rollbackHandler's execution roles, or actions work but rollbacks silently AccessDenied (or vice versa).
# Skip entirely for a single-account deployment.
TARGET_ACCOUNT_PROFILE = None  # set to an AWS CLI profile name for a second onboarded account, else leave None
if TARGET_ACCOUNT_PROFILE:
    p = sh(f'aws iam get-role --role-name "opencdr-$OPENCDR_STAGE-ir-role" '
           f'--profile {TARGET_ACCOUNT_PROFILE} --query "Role.AssumeRolePolicyDocument" --output json')
    trust = json.loads(p.stdout)
    principals = json.dumps(trust)
    has_responder = "responder" in principals
    has_rollback = "rollbackHandler" in principals
    ok = has_responder and has_rollback
    record("P3-IR", "multi-account IR role trusts both responder+rollbackHandler",
           "PASS" if ok else "FAIL", f"responder={has_responder} rollbackHandler={has_rollback}")
else:
    record("P3-IR", "multi-account IR role trust policy", "SKIP", "single-account deployment")


---
## Phase 4 — P2 (High): API Surface, Auth & the MCP Server

The management surface a client will actually use day to day. See
[`docs/api-reference.md`](../docs/api-reference.md) and [`docs/security.md`](../docs/security.md).


In [ ]:
# 4.1 -- API key scope enforcement: a read-only key must get 403 on a mutating route.
ro_key = os.environ.get("OPENCDR_READONLY_API_KEY", "")
if ro_key:
    status, body = api("POST", "/rules", body={"rule_id": "zzz_scope_probe", "rule_kind": "signal",
                                                "enabled": False, "conditions": []},
                        key=ro_key, expect=403)
    ok = status == 403 and "scope" in json.dumps(body).lower()
    record("P4-API", "read-only key rejected on mutating route", "PASS" if ok else "FAIL",
           f"status={status} body={body}")
else:
    record("P4-API", "read-only key rejected on mutating route", "SKIP",
           "set OPENCDR_READONLY_API_KEY in Phase 0 to run this -- see docs/api-reference.md#api-key-scopes")


In [ ]:
# 4.2 -- Secrets are never returned in plaintext, even to the key that set them.
status, settings = api("GET", "/settings", expect=200, quiet=True)
secret_fields_present = any(k in settings for k in ("slack", "discord", "jira", "webhook"))
redacted_ok = True
for k in ("slack", "discord"):
    v = settings.get(k) or {}
    url = v.get("webhook_url") or v.get("url")
    if url and url != "***REDACTED***":
        redacted_ok = False
record("P4-API", "secrets masked on GET /settings", "PASS" if redacted_ok else "FAIL",
       "no secret-shaped fields configured yet" if not secret_fields_present else "")


In [ ]:
# 4.3 -- Pagination: page_size=1 + next_token should walk distinct items, not repeat/lose one.
status, page1 = api("GET", "/signals?severity=HIGH&page_size=1", expect=200, quiet=True)
items1 = page1.get("items", [])
token = page1.get("next_token")
if items1 and token:
    status, page2 = api("GET", f"/signals?severity=HIGH&page_size=1&next_token={token}", expect=200, quiet=True)
    items2 = page2.get("items", [])
    distinct = bool(items2) and items2[0].get("detection_id") != items1[0].get("detection_id")
    record("P4-API", "pagination returns distinct pages", "PASS" if distinct else "FAIL")
else:
    record("P4-API", "pagination returns distinct pages", "SKIP", "fewer than 2 HIGH signals in range to page across")


In [ ]:
# 4.4 -- /signals/stats (documented in this repo's docs, NOT in openapi.yml -- see docs/api-reference.md)
status, stats = api("GET", "/signals/stats", expect=200)
print(json.dumps(stats, indent=2))
six_severities = {"CRITICAL", "HIGH", "MEDIUM", "LOW", "INFO", "INFORMATIONAL"}
ok = status == 200 and six_severities.issubset(set(stats.get("counts", {}).keys()))
record("P4-API", "/signals/stats returns all severities (even at 0)", "PASS" if ok else "FAIL")


In [ ]:
# 4.5 -- IR roles CRUD smoke test against a scratch, obviously-fake account ID -- never resolves
# to a real assumable role, safe to create/mutate/delete freely.
SCRATCH_ACCOUNT = "000000000000"
status, _ = api("POST", "/ir-roles", body={"aws_account_id": SCRATCH_ACCOUNT,
                                            "role_arn": f"arn:aws:iam::{SCRATCH_ACCOUNT}:role/scratch-probe"},
                 expect=200)
status_g, got = api("GET", f"/ir-roles/{SCRATCH_ACCOUNT}", expect=200)
status_p, _ = api("PUT", f"/ir-roles/{SCRATCH_ACCOUNT}",
                   body={"role_arn": got.get("role_arn"), "enabled": False}, expect=200)
status_d, _ = api("DELETE", f"/ir-roles/{SCRATCH_ACCOUNT}", expect=200)
ok = 200 in (status, status_g, status_p, status_d)
record("P4-API", "IR roles CRUD (create/get/update/delete)",
       "PASS" if all(s == 200 for s in (status, status_g, status_p, status_d)) else "FAIL")


In [ ]:
# 4.6 -- MCP server: needs an actual MCP-aware client, can't be driven headlessly from here.
sh("pip show mcp 2>/dev/null | head -3", check_rc=False)
manual_check("P4-API", "MCP server connects and serves tools",
             "claude mcp add opencdr --env OPENCDR_API_URL=... --env OPENCDR_API_KEY=<mcp-scoped key> "
             "-- python mcp_server/server.py, then: run the opencdr_status tool (should match "
             "Phase 1.1), then one read tool per surface (opencdr_rules_list, opencdr_signals_search, "
             "opencdr_settings_get). The server defaults to the read-only 'observer' profile: "
             "mutating/destructive tools (e.g. opencdr_rules_delete, opencdr_ir_actions_rollback) "
             "are NOT registered unless OPENCDR_MCP_PROFILE=operator|responder is set AND paired "
             "with a matching scoped key. See docs/api-reference.md#mcp-server-default-management-plane.")

---
## Phase 5 — P2 (High): Detection Rule Coverage

Phase 1 proved the pipeline works. This phase proves *breadth* — every bundled rule, correlation
rules specifically (a separate code path from signal rules — `alerter`, not `processor`), list
rules, and a custom rule authored at runtime. See
[`docs/detection-rules.md`](../docs/detection-rules.md).


In [ ]:
# 5.1 -- Full local rule suite against every fixture, no AWS calls (fast, safe to re-run any time)
result = sh("python3 scripts/test_rules_local.py", check_rc=False)
counts = count_markers(result.stdout)
ok = counts["FAIL"] == 0
record("P5-Rules", "local rule suite (test_rules_local.py)", "PASS" if ok else "FAIL", str(counts))


In [ ]:
# 5.2 -- Correlation rules run in `alerter`, a separate path from signal rules in `processor`.
# There's no public /alerts endpoint (see docs/api-reference.md) -- alerter also enqueues the
# correlation result as its own signal, and emits a CorrelationMatches EMF metric. Check both.
sh('python3 scripts/opencdr.py test deployed --stage "$OPENCDR_STAGE" --event 029', check_rc=False)
sh('python3 scripts/opencdr.py test deployed --stage "$OPENCDR_STAGE" --event 030', check_rc=False)
time.sleep(10)  # let alerter's DynamoDB-stream trigger run
p = sh('aws cloudwatch get-metric-statistics --namespace OpenCDR --metric-name CorrelationMatches '
       '--start-time "$(date -u -v-10M +%Y-%m-%dT%H:%M:%S 2>/dev/null || date -u -d \'10 minutes ago\' +%Y-%m-%dT%H:%M:%S)" '
       '--end-time "$(date -u +%Y-%m-%dT%H:%M:%S)" --period 300 --statistics Sum --region "$AWS_REGION"',
       check_rc=False)
manual_check("P5-Rules", "correlation rules produce CorrelationMatches",
             "The CloudWatch metric query above returns a non-empty Datapoints list with Sum >= 1 "
             "for the last 10 minutes (EMF metrics can lag a minute or two behind the log line).")


In [ ]:
# 5.3 -- List rules (allow-lists) -- a third rule_kind, used by in_list/not_in_list conditions.
sh('python3 scripts/opencdr.py lists create manual-test-list --description "notebook smoke test" --values 1.2.3.4')
sh('python3 scripts/opencdr.py lists add manual-test-list 5.6.7.8')
out = sh('python3 scripts/opencdr.py lists show manual-test-list')
ok = "1.2.3.4" in out.stdout and "5.6.7.8" in out.stdout
sh('python3 scripts/opencdr.py lists remove manual-test-list 5.6.7.8', check_rc=False)
sh('python3 scripts/opencdr.py lists delete manual-test-list', check_rc=False)
record("P5-Rules", "list rules CRUD (create/add/show/remove/delete)", "PASS" if ok else "FAIL")


In [ ]:
# 5.4 -- Custom rule authored live via the API (not via load_rules.sh), then cleaned up.
status, _ = api("POST", "/rules", body={
    "rule_id": "999_notebook_smoke_test", "rule_kind": "signal", "enabled": True,
    "severity": "LOW", "notify": False,
    "conditions": [{"field": "activity_name", "op": "equals", "value": "PutBucketPolicy"}],
}, expect=200)
status_g, got = api("GET", "/rules/999_notebook_smoke_test?rule_kind=signal", expect=200)
status_d, _ = api("DELETE", "/rules/999_notebook_smoke_test?rule_kind=signal", expect=200)
ok = status == 200 and status_g == 200 and got.get("severity") == "LOW" and status_d == 200
record("P5-Rules", "custom rule CRUD via API", "PASS" if ok else "FAIL")


---
## Phase 6 — P3 (Medium): Multi-Region, Multi-Account & Org Coverage

**Skip this entire phase for a genuinely single-account, single-region deployment** — a fresh
deploy already covers that case with zero extra setup. Relevant docs:
[`region-forwarding.md`](../docs/region-forwarding.md), [`ir-role.md`](../docs/ir-role.md),
[`org-forwarding.md`](../docs/org-forwarding.md).


In [ ]:
# 6.1 -- Cross-region forwarding: dry-run preview is always safe to run.
ADDITIONAL_REGION = None  # e.g. "eu-west-1" -- leave None to skip this phase
if ADDITIONAL_REGION:
    sh(f'./scripts/setup_region_forwarding.sh --stage "$OPENCDR_STAGE" --region {ADDITIONAL_REGION} --dry-run')
    manual_check("P6-Multi", "cross-region forwarding delivers events",
                 f"After running setup_region_forwarding.sh for real (no --dry-run) against "
                 f"{ADDITIONAL_REGION}, an action performed in that region (e.g. a console login) "
                 f"produces a signal in the home region within ~1 minute.")
else:
    record("P6-Multi", "cross-region forwarding", "SKIP", "single-region deployment")


In [ ]:
# 6.2 -- Multi-account IR role onboarding: full walkthrough is docs/ir-role.md; this just
# confirms the mapping resolves once created (safe with the scratch account from 4.5's pattern,
# or a real second account you own).
SECOND_ACCOUNT_ID = None  # set to a real AWS account ID you've onboarded, else leave None
if SECOND_ACCOUNT_ID:
    status, mapping = api("GET", f"/ir-roles/{SECOND_ACCOUNT_ID}", expect=200)
    print(json.dumps(mapping, indent=2))
    record("P6-Multi", "second account IR role mapping resolves", "PASS" if status == 200 else "FAIL")
else:
    record("P6-Multi", "multi-account IR role onboarding", "SKIP", "single-account deployment")


In [ ]:
# 6.3 -- Org-wide forwarding: dry-run preview only (real onboarding needs per-member-account
# credentials this notebook doesn't assume you have handy).
ORG_ID = None  # e.g. "o-xxxxxxxxxx" -- leave None to skip
if ORG_ID:
    sh(f'./scripts/setup_org_forwarding.sh --stage "$OPENCDR_STAGE" --profiles <member-a,member-b> --dry-run')
    manual_check("P6-Multi", "org-wide forwarding", "Central account deployed with --param=\"orgId=" + ORG_ID + "\", each member onboarded, and a forged PutEvents call from a non-forwarder principal is rejected (see docs/org-forwarding.md's threat model).")
else:
    record("P6-Multi", "org-wide forwarding", "SKIP", "not an AWS Organizations deployment")


---
## Phase 7 — P3 (Medium): Observability

What tells you *OpenCDR itself* is healthy (as opposed to what it's detecting). See
[`docs/observability.md`](../docs/observability.md).


In [ ]:
# 7.1 -- Dashboard exists (auto-provisioned on every deploy, zero config)
p = sh(f'aws cloudformation describe-stacks --stack-name "opencdr-$OPENCDR_STAGE" --region "$AWS_REGION" '
       f'--query "Stacks[0].Outputs[?OutputKey==\'DashboardUrl\'].OutputValue" --output text', check_rc=False)
dashboard_url = p.stdout.strip()
print("Dashboard:", dashboard_url)
manual_check("P7-Observability", "CloudWatch dashboard shows healthy widgets",
             f"Open {dashboard_url}: Lambda errors flat, p99 duration reasonable, all 4 queue "
             "depths near zero (notifications-dlq/responses-dlq/stream-failures/signals-write-dlq), "
             "and the 5 custom-metric SEARCH widgets show data from Phase 1/5's activity.")


In [ ]:
# 7.2 -- Custom EMF metrics exist under the OpenCDR namespace (docs/observability.md#custom-metrics)
p = sh('aws cloudwatch list-metrics --namespace OpenCDR --region "$AWS_REGION" --query "Metrics[].MetricName" --output json')
present = set(json.loads(p.stdout))
expected = {"SignalsCreated", "CorrelationMatches", "PublishSuccess", "PublishFailure", "ResponderActionsExecuted"}
missing = expected - present
record("P7-Observability", "expected custom metrics present", "PASS" if not missing else "WARN",
       f"missing: {missing}" if missing else "all 5 metric names seen at least once")


In [ ]:
# 7.3 -- All 16 CloudWatch alarms exist (see docs/observability.md for the exact breakdown)
p = sh(f'aws cloudwatch describe-alarms --alarm-name-prefix "opencdr-$OPENCDR_STAGE" --region "$AWS_REGION" '
       f'--query "MetricAlarms[].{{Name:AlarmName,State:StateValue}}" --output table')
n_alarms_p = sh(f'aws cloudwatch describe-alarms --alarm-name-prefix "opencdr-$OPENCDR_STAGE" --region "$AWS_REGION" '
                f'--query "length(MetricAlarms)" --output text', quiet=True)
n_alarms = int(n_alarms_p.stdout.strip() or 0)
record("P7-Observability", "16 alarms provisioned", "PASS" if n_alarms >= 16 else "WARN", f"found {n_alarms}")


In [ ]:
# 7.4 -- Alarm delivery: is anything actually subscribed to AlarmsSnsTopic?
p = sh(f'aws cloudformation describe-stacks --stack-name "opencdr-$OPENCDR_STAGE" --region "$AWS_REGION" '
       f'--query "Stacks[0].Outputs[?OutputKey==\'AlarmsTopicArn\'].OutputValue" --output text', check_rc=False)
alarms_topic = p.stdout.strip()
if alarms_topic:
    subs = sh(f'aws sns list-subscriptions-by-topic --topic-arn "{alarms_topic}" --region "$AWS_REGION" '
              f'--query "Subscriptions[].{{Protocol:Protocol,Endpoint:Endpoint,Status:SubscriptionArn}}" --output table')
    has_confirmed = "PendingConfirmation" not in subs.stdout and subs.stdout.strip() != ""
    record("P7-Observability", "alarm delivery configured & confirmed",
           "PASS" if has_confirmed else "WARN",
           "check for PendingConfirmation subscriptions (click the confirmation email) or set up "
           "alarmNotifier's Slack SSM param -- see docs/observability.md#alarms-exist-automatically")
else:
    record("P7-Observability", "alarm delivery configured & confirmed", "FAIL", "AlarmsTopicArn output not found")


In [ ]:
# 7.5 -- X-Ray traces show DynamoDB/SQS as their own nodes, not just Lambda invocation boundaries
manual_check("P7-Observability", "X-Ray service map",
             "X-Ray console (same account/region) shows DynamoDB and SQS nodes in the service map "
             "for the last hour, not just Lambda nodes -- confirms src/infra/xray_setup.py's "
             "patch(['boto3']) is actually wired, not just the invocation-boundary tracing.")


In [ ]:
# 7.6 -- Cost tracking (requires the two one-time Billing-console steps -- see docs/setup.md#6)
sh('./scripts/cost_report.sh --stage "$OPENCDR_STAGE"', check_rc=False)
manual_check("P7-Observability", "cost report returns real numbers",
             "cost_report.sh above returns non-zero spend broken out by this stack's Project/Stage "
             "tags (needs Cost Explorer enabled + tags activated as cost allocation tags, up to 24h "
             "after activation -- see docs/setup.md step 6).")


---
## Phase 8 — P4 (Low): Data Archival & Retention

Confirms the 90-day DynamoDB TTL is genuinely "archive first, then expire," not silent data loss.
See [`docs/data-archival.md`](../docs/data-archival.md).


In [ ]:
# 8.1 -- TTL enabled on all four archived-then-expired tables
for table in ("signals-table-v2", "alerts-table", "logs-table-v2", "outbox-table"):
    full_name = f'opencdr-{os.environ["OPENCDR_STAGE"]}-{table}'
    p = sh(f'aws dynamodb describe-time-to-live --table-name "{full_name}" --region "$AWS_REGION" '
           f'--query "TimeToLiveDescription.TimeToLiveStatus" --output text', check_rc=False, quiet=True)
    status_str = p.stdout.strip()
    print(f"{full_name}: {status_str}")
    record("P8-Archival", f"TTL enabled on {table}", "PASS" if status_str == "ENABLED" else "FAIL", status_str)


In [ ]:
# 8.2 -- archiver actually processed the Phase 1 canary signal (fast, deterministic proxy for
# "the DynamoDB Streams -> archiver -> Firehose leg works", per docs/data-archival.md's own CI check)
if CANARY_DETECTION_ID:
    time.sleep(5)
    p = sh(f'aws logs filter-log-events --log-group-name "/aws/lambda/opencdr-$OPENCDR_STAGE-archiver" '
           f'--region "$AWS_REGION" --filter-pattern "\\"{CANARY_DETECTION_ID}\\"" '
           f'--query "events[].message" --output text', check_rc=False)
    found = CANARY_DETECTION_ID in p.stdout
    record("P8-Archival", "archiver processed the canary signal", "PASS" if found else "WARN",
           "not found yet -- archiver's DynamoDB-stream trigger can lag a few seconds; re-run this cell")
else:
    record("P8-Archival", "archiver processed the canary signal", "SKIP", "no canary captured in Phase 1.5")


In [ ]:
# 8.3 -- Parquet actually lands in S3 and is queryable via Athena. Firehose buffers up to
# 128MB/300s, so this needs a real wait -- not worth gating routine test runs on, per
# docs/data-archival.md's own "not verified by CI" note. Manual, once per deployment.
manual_check("P8-Archival", "Athena can query the archived canary",
             "Wait ~5 minutes after 8.2 passes, then in Athena: SELECT * FROM signals WHERE "
             f"account='<account_id>' AND year='...' AND month='...' AND day='...' AND "
             f"detection_id='{CANARY_DETECTION_ID if 'CANARY_DETECTION_ID' in dir() else '<id>'}' "
             "returns exactly one row. Remember: account/year/month/day are equality-filtered "
             "partition-projection columns, not optional.")


---
## Phase 9 — P4 (Low): Security & IAM Boundary Spot Checks

Confirms the documented IAM model actually holds in this deployment, not just in
[`docs/security.md`](../docs/security.md)'s prose.


In [ ]:
# 9.1 -- responder's OWN execution role must have zero IAM/EC2/S3 grants -- all destructive
# capability must come from the ASSUMED role, never responder's own credentials.
role_p = sh(f'aws lambda get-function-configuration --function-name "opencdr-$OPENCDR_STAGE-responder" '
            f'--region "$AWS_REGION" --query Role --output text', quiet=True)
role_arn = role_p.stdout.strip()
role_name = role_arn.rsplit("/", 1)[-1]
inline = sh(f'aws iam list-role-policies --role-name "{role_name}" --output json', quiet=True)
attached = sh(f'aws iam list-attached-role-policies --role-name "{role_name}" --output json', quiet=True)
print(inline.stdout, attached.stdout)
dangerous = any(x in (inline.stdout + attached.stdout) for x in ("IAMFullAccess", "AdministratorAccess"))
record("P9-Security", "responder's own role has no destructive grants", "WARN" if dangerous else "PASS",
       "inspect the inline policy documents above for iam:/ec2:/s3: actions -- only sts:AssumeRole "
       "on opencdr-<stage>-ir-role should appear")


In [ ]:
# 9.2 -- Every Lambda has its OWN execution role (provider.iam.role.mode: perFunction), not one
# shared role for the whole stack.
fns = ["processor", "signalWriter", "alerter", "publisher", "notifier", "responder",
       "rollbackHandler", "api", "alarmNotifier", "archiver"]
roles = {}
for fn in fns:
    p = sh(f'aws lambda get-function-configuration --function-name "opencdr-$OPENCDR_STAGE-{fn}" '
           f'--region "$AWS_REGION" --query Role --output text', check_rc=False, quiet=True)
    roles[fn] = p.stdout.strip()
print(json.dumps(roles, indent=2))
distinct = len(set(roles.values())) == len([v for v in roles.values() if v])
record("P9-Security", "every Lambda has a distinct execution role", "PASS" if distinct else "FAIL")


In [ ]:
# 9.3 -- API Gateway usage plan matches the documented backstop (10,000 req/month, 100 rps)
p = sh('aws apigateway get-usage-plans --region "$AWS_REGION" '
       '--query "items[?contains(name, \'opencdr\')].{{Name:name,Quota:quota,Throttle:throttle}}" --output json'
       .format(), check_rc=False)
print(p.stdout)
manual_check("P9-Security", "usage plan quota/throttle as documented",
             "quota.limit == 10000 (MONTH) and throttle.rateLimit == 100, per docs/security.md#api-gateway.")


---
## Phase 10 — P4 (Optional): SIEM Fan-Out

Only relevant if this deployment forwards alerts to an external SIEM. See
[`docs/siem-integrations.md`](../docs/siem-integrations.md) and
[`docs/notifications.md#sns-fan-out-for-anything-the-built-in-channels-dont-cover`](../docs/notifications.md).


In [ ]:
SIEM_INTEGRATION_IN_USE = False  # flip to True if this deployment forwards to a SIEM
if SIEM_INTEGRATION_IN_USE:
    manual_check("P10-SIEM", "SIEM receives forwarded alerts",
                 "Trigger a fixture (e.g. --event 001) and confirm the alert arrives in the SIEM "
                 "(Datadog/Splunk/Sentinel/Elastic/QRadar/Sumo Logic) via whichever of the "
                 "AlertsSnsTopic fan-out or custom-webhook channel this deployment uses.")
else:
    record("P10-SIEM", "SIEM fan-out", "SKIP", "no SIEM integration configured for this deployment")


---
## Sign-off Summary

Run this last. It aggregates every `record(...)` call above — including any `MANUAL-PENDING`
checks you haven't resolved yet by editing their cell and re-running it.


In [ ]:
from collections import Counter

by_phase = {}
for r in RESULTS:
    by_phase.setdefault(r["phase"], []).append(r)

print(f"{'Phase':<18} {'PASS':>5} {'FAIL':>5} {'WARN':>5} {'SKIP':>5} {'PENDING':>8}")
print("-" * 55)
totals = Counter()
for phase, rows in by_phase.items():
    c = Counter(r["status"] for r in rows)
    totals.update(c)
    print(f"{phase:<18} {c['PASS']:>5} {c['FAIL']:>5} {c['WARN']:>5} {c['SKIP']:>5} {c['MANUAL-PENDING']:>8}")
print("-" * 55)
print(f"{'TOTAL':<18} {totals['PASS']:>5} {totals['FAIL']:>5} {totals['WARN']:>5} {totals['SKIP']:>5} {totals['MANUAL-PENDING']:>8}")

print()
if totals["FAIL"] > 0:
    print("❌ NOT READY -- one or more FAIL checks above must be resolved.")
elif totals["MANUAL-PENDING"] > 0:
    print(f"\U0001f464 {totals['MANUAL-PENDING']} manual check(s) still unresolved -- verify by hand and re-run their cells.")
else:
    print("✅ All automated checks passed and no manual checks are pending.")

print("\nFailures and warnings for quick triage:")
for r in RESULTS:
    if r["status"] in ("FAIL", "WARN"):
        print(f"  [{r['phase']}] {r['check']}: {r['status']} -- {r['detail']}")


### Tester sign-off

| Field | Value |
|---|---|
| Tester | *fill in* |
| Date | *fill in* |
| AWS account / stage / region | *fill in* |
| Phases skipped and why | *fill in* |
| Overall verdict | *fill in* |
| Notes / follow-ups filed | *fill in* |

**Related pages:** [`docs/manual-testing.md`](../docs/manual-testing.md) (condensed runbook this
notebook expands) · [`docs/setup.md`](../docs/setup.md) (first-time deployment) ·
[`docs/architecture.md`](../docs/architecture.md) · [`docs/security.md`](../docs/security.md)
